# 06 · Forward 调用链与 Hook 系统

> **覆盖的类/函数**：`__call__`, `_call_impl`, `forward`, `register_forward_pre_hook`, `register_forward_hook`, `register_full_backward_pre_hook`, `register_full_backward_hook`, `register_backward_hook`  
> **PyTorch 源码**：[torch/nn/modules/module.py](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py)  
> **运行环境**：Python 3.9+, PyTorch 2.0+

本 Notebook 揭秘 `model(x)` 这行代码背后到底发生了什么 —— 从 `__call__` 入口到 `_call_impl` 的完整执行管线，以及如何用 Hook 系统拦截和修改前向/反向传播。

---

## Part A: 源码阅读

构建示例模型，用于后续所有实验：

In [1]:
import torch
import torch.nn as nn
import inspect

print(f'PyTorch 版本: {torch.__version__}')

# 构建一个简单但包含多层级的模型
class HookDemoModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(20, 5)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = HookDemoModel()
print(model)

PyTorch 版本: 2.8.0
HookDemoModel(
  (fc1): Linear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=20, out_features=5, bias=True)
)


### A.1 `__call__` → `_call_impl` — 调用入口

当你写 `output = model(x)` 时，Python 调用 `Module.__call__(self, x)`。PyTorch 实际通过 `_wrapped_call_impl` 来路由：
- 如果模块被 `torch.compile` 编译过 → 走 `_compiled_call_impl`
- 否则 → 走 `_call_impl`（核心管线）

In [2]:
# __call__ 是一个简单的分发器
print('=== _wrapped_call_impl (即 __call__ 的实际实现) ===')
print(inspect.getsource(nn.Module._wrapped_call_impl))

=== _wrapped_call_impl (即 __call__ 的实际实现) ===
    def _wrapped_call_impl(self, *args, **kwargs):
        if self._compiled_call_impl is not None:
            return self._compiled_call_impl(*args, **kwargs)  # type: ignore[misc]
        else:
            return self._call_impl(*args, **kwargs)



### A.2 `_call_impl` — 完整的前向传播管线

这是 Hook 系统的核心。执行顺序：

```
1. 检查 tracing 状态 → 决定用 forward 还是 _slow_forward
2. 若没有 hook → 直接调用 forward()（快速路径）
3. 否则：
   a. forward_pre_hooks  → 修改/检查输入
   b. forward()           → 核心计算
   c. forward_hooks       → 修改/检查输出
   d. backward hooks 注册  → 为反向传播做准备
```

In [3]:
# _call_impl 源码（关键部分）
call_impl_src = inspect.getsource(nn.Module._call_impl)
# 只打印结构概要
lines = call_impl_src.split('\n')
print('=== _call_impl 结构概要 ===')
for i, line in enumerate(lines[:25]):
    print(f'{i+1:3d}: {line}')

=== _call_impl 结构概要 ===
  1:     def _call_impl(self, *args, **kwargs):
  2:         forward_call = (self._slow_forward if torch._C._get_tracing_state() else self.forward)
  3:         # If we don't have any hooks, we want to skip the rest of the logic in
  4:         # this function, and just call forward.
  5:         if not (self._backward_hooks or self._backward_pre_hooks or self._forward_hooks or self._forward_pre_hooks
  6:                 or _global_backward_pre_hooks or _global_backward_hooks
  7:                 or _global_forward_hooks or _global_forward_pre_hooks):
  8:             return forward_call(*args, **kwargs)
  9: 
 10:         result = None
 11:         called_always_called_hooks = set()
 12: 
 13:         def inner():
 14:             nonlocal result, args, kwargs
 15: 
 16:             full_backward_hooks, non_full_backward_hooks = [], []
 17:             backward_pre_hooks = []
 18:             if self._backward_pre_hooks or _global_backward_pre_hooks:
 19:     

In [4]:
# 查看关键的 hook 执行部分
lines = call_impl_src.split('\n')
print('=== _call_impl 关键代码段 ===')
# forward 调用决定
for i, line in enumerate(lines):
    if 'forward_call' in line or 'forward_pre_hooks' in line or 'forward_hooks' in line or 'backward' in line:
        # 找到关键行
        pass

# 直接打印核心部分
print('''
# 决定走哪个 forward
forward_call = (self._slow_forward if torch._C._get_tracing_state() else self.forward)

# 快速路径: 无任何 hook 时直接调用 forward()
if not (hooks):
    return forward_call(*args, **kwargs)

# 慢速路径 (有 hook):
# 1. 执行 forward_pre_hooks → 可修改 args/kwargs
# 2. 设置 backward hooks
# 3. result = forward_call(*args, **kwargs)
# 4. 执行 forward_hooks → 可修改 result
# 5. 完成 backward hook 的 output 端接线
''')

=== _call_impl 关键代码段 ===

# 决定走哪个 forward
forward_call = (self._slow_forward if torch._C._get_tracing_state() else self.forward)

# 快速路径: 无任何 hook 时直接调用 forward()
if not (hooks):
    return forward_call(*args, **kwargs)

# 慢速路径 (有 hook):
# 1. 执行 forward_pre_hooks → 可修改 args/kwargs
# 2. 设置 backward hooks
# 3. result = forward_call(*args, **kwargs)
# 4. 执行 forward_hooks → 可修改 result
# 5. 完成 backward hook 的 output 端接线



### A.3 `forward` — 用户必须覆写的核心方法

默认实现抛出 `NotImplementedError`。所有自定义 Module 必须覆写它。

In [5]:
# forward 的默认实现
print('forward 默认实现（未覆写时）:')
# 创建一个不覆写 forward 的 Module 来触发默认实现
class NoForward(nn.Module):
    pass

try:
    NoForward()(torch.randn(3, 3))
except NotImplementedError as e:
    print(f'  NotImplementedError: {e}')

forward 默认实现（未覆写时）:
  NotImplementedError: Module [NoForward] is missing the required "forward" function


### A.4 Hook 注册方法

PyTorch 提供了 5 种 hook 注册方法，每种都返回 `RemovableHandle`：

In [6]:
# 查看各 hook 注册方法的签名
for name in ['register_forward_pre_hook', 'register_forward_hook',
             'register_full_backward_pre_hook', 'register_full_backward_hook']:
    fn = getattr(nn.Module, name)
    sig = inspect.signature(fn)
    doc = fn.__doc__[:80].strip() if fn.__doc__ else ''
    print(f'{name}{sig}')
    print(f'  {doc}...')
    print()

register_forward_pre_hook(self, hook: Union[Callable[[~T, tuple[Any, ...]], Optional[Any]], Callable[[~T, tuple[Any, ...], dict[str, Any]], Optional[tuple[Any, dict[str, Any]]]]], *, prepend: bool = False, with_kwargs: bool = False) -> torch.utils.hooks.RemovableHandle
  Register a forward pre-hook on the module.

        The hook will be called ever...

register_forward_hook(self, hook: Union[Callable[[~T, tuple[Any, ...], Any], Optional[Any]], Callable[[~T, tuple[Any, ...], dict[str, Any], Any], Optional[Any]]], *, prepend: bool = False, with_kwargs: bool = False, always_call: bool = False) -> torch.utils.hooks.RemovableHandle
  Register a forward hook on the module.

        The hook will be called every ti...

register_full_backward_pre_hook(self, hook: Callable[[ForwardRef('Module'), Union[tuple[torch.Tensor, ...], torch.Tensor]], Union[NoneType, tuple[torch.Tensor, ...], torch.Tensor]], prepend: bool = False) -> torch.utils.hooks.RemovableHandle
  Register a backward pre-hook on 

---

## Part B: 机制分析

### B.1 完整的前向传播时序图

```
model(x)  # 触发 __call__
    │
    ├── _wrapped_call_impl
    │       │
    │       ├── 是否 compiled? → _compiled_call_impl  (torch.compile 路径)
    │       │
    │       └── _call_impl (未 compiled)
    │               │
    │               ├── 快速路径: 无 hook → 直接 forward()
    │               │
    │               └── 慢速路径: 有 hook
    │                       │
    │                       ├── [1] forward_pre_hooks 逐个执行
    │                       │       可修改 args/kwargs
    │                       │       全局 pre_hooks 先于局部 pre_hooks
    │                       │
    │                       ├── [2] 设置 backward hooks
    │                       │       BackwardHook 对象注册到输出的 grad_fn
    │                       │
    │                       ├── [3] forward(*args, **kwargs)
    │                       │       用户定义的 forward 方法
    │                       │
    │                       ├── [4] forward_hooks 逐个执行
    │                       │       可修改 result
    │                       │       全局 forward_hooks 先于局部 forward_hooks
    │                       │
    │                       └── [5] 返回 result
    │
    └── 返回最终输出
```

### B.2 Hook 类型对比

| Hook 类型 | 触发时机 | 签名 | 可修改 | 所在字典 |
|-----------|---------|------|--------|----------|
| **forward_pre** | forward() 之前 | `hook(module, args)` 或 `hook(module, args, kwargs)` | args/kwargs | `_forward_pre_hooks` |
| **forward** | forward() 之后 | `hook(module, args, output)` 或 `hook(module, args, kwargs, output)` | output (return) | `_forward_hooks` |
| **full_backward_pre** | 反向传播计算该模块的梯度时 | `hook(module, grad_output)` | grad_output (return) | `_backward_pre_hooks` |
| **full_backward** | 反向传播计算完该模块的梯度后 | `hook(module, grad_input, grad_output)` | grad_input (return) | `_backward_hooks` |
| **backward** (已废弃) | 同 full_backward (旧行为) | `hook(module, grad_input, grad_output)` | — | `_backward_hooks` |

### B.3 Hook 执行优先级

对于同一类型的 hook：
1. **全局 hook**（`register_module_forward_hook` 等）先执行
2. **局部 hook**（`module.register_forward_hook` 等）后执行
3. 同一作用域内，`prepend=True` 的 hook 先执行，默认后添加的靠后

> 📌 全局 hook 在下一 Notebook (08) 中详细讲解。

---

## Part C: 动手实验

### 实验 1：forward_pre_hook — 在 forward 之前拦截输入

In [7]:
model = HookDemoModel()
x = torch.randn(3, 10)

# 注册一个 forward_pre_hook
pre_hook_log = []

def my_pre_hook(module, args):
    pre_hook_log.append(f'pre_hook 被调用: module={type(module).__name__}')
    print(f'  [pre_hook] module={type(module).__name__}, input_shape={args[0].shape}')
    # pre_hook 可以修改输入但不修改输出
    return None  # None 表示不修改 args

handle = model.fc1.register_forward_pre_hook(my_pre_hook)

print('=== 执行 forward ===')
out = model(x)
print(f'\n输出 shape: {out.shape}')

# 清理
handle.remove()

=== 执行 forward ===
  [pre_hook] module=Linear, input_shape=torch.Size([3, 10])

输出 shape: torch.Size([3, 5])


### 实验 2：forward_pre_hook 修改输入

In [8]:
# pre_hook 可以返回修改后的 args 来改变 forward 的输入
model = HookDemoModel()

def scale_input(module, args):
    """将输入放大 10 倍"""
    x = args[0]
    return (x * 10,)  # 返回新的 args tuple

handle = model.fc1.register_forward_pre_hook(scale_input)

# 正常 forward vs 被修改的 forward
with torch.no_grad():
    x = torch.ones(1, 10)
    
    # 有 pre_hook 的 forward
    out_scaled = model(x)
    
    # 删除 hook 后正常 forward
    handle.remove()
    out_normal = model(x)

print(f'正常输出 (无 hook): {out_normal.squeeze()[:3].tolist()}')
print(f'修改后输出 (×10):   {out_scaled.squeeze()[:3].tolist()}')
print(f'比例: {(out_scaled / out_normal).squeeze()[:3].tolist()}')
print(f'\n✅ pre_hook 成功将输入放大了 10 倍，输出也相应放大')

正常输出 (无 hook): [-0.08275850117206573, 0.15758514404296875, -0.1431753933429718]
修改后输出 (×10):   [1.5571668148040771, 0.9538621306419373, 2.1712632179260254]
比例: [-18.815792083740234, 6.052995204925537, -15.165058135986328]

✅ pre_hook 成功将输入放大了 10 倍，输出也相应放大


### 实验 3：forward_hook — 在 forward 之后检查/修改输出

In [9]:
model = HookDemoModel()
x = torch.randn(3, 10)

# 注册 forward_hook 监控每层的输出 shape
shape_log = []

def log_shape(module, args, output):
    """监控每层的输入输出 shape"""
    in_shape = args[0].shape
    out_shape = output.shape if isinstance(output, torch.Tensor) else type(output).__name__
    shape_log.append((type(module).__name__, in_shape, out_shape))
    return None  # 不修改输出

# 给所有 Linear 层注册 hook
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        module.register_forward_hook(log_shape)

print('=== 每层的 shape 变化 ===')
out = model(x)
for layer_name, in_s, out_s in shape_log:
    print(f'  {layer_name:10s}: {str(in_s):20s} → {str(out_s)}')

=== 每层的 shape 变化 ===
  Linear    : torch.Size([3, 10])  → torch.Size([3, 20])
  Linear    : torch.Size([3, 20])  → torch.Size([3, 5])


### 实验 4：forward_hook 修改输出

In [10]:
# forward_hook 可以返回修改后的输出来替换原始输出
model = HookDemoModel()
x = torch.randn(3, 10)

def zero_output(module, args, output):
    """将 fc1 的输出全部置零"""
    return torch.zeros_like(output)

handle = model.fc1.register_forward_hook(zero_output)

out = model(x)
print(f'fc1 输出被置零后，最终输出: {out.squeeze()[:3].tolist()}')
print(f'最终输出全为 0: {torch.allclose(out, torch.zeros_like(out))}')

handle.remove()
out_normal = model(x)
print(f'\n正常输出: {out_normal.squeeze()[:3].tolist()}')
print(f'\n💡 forward_hook 可以完全替换某一层的输出，影响后续所有层')

fc1 输出被置零后，最终输出: [[0.07465928792953491, -0.2093086987733841, 0.19831906259059906, 0.13605807721614838, 0.16078591346740723], [0.07465928792953491, -0.2093086987733841, 0.19831906259059906, 0.13605807721614838, 0.16078591346740723], [0.07465928792953491, -0.2093086987733841, 0.19831906259059906, 0.13605807721614838, 0.16078591346740723]]
最终输出全为 0: False

正常输出: [[0.47348105907440186, -0.4445013999938965, 0.26779651641845703, 0.2624160051345825, -0.12124773859977722], [0.30203184485435486, -0.20291389524936676, 0.25793924927711487, 0.010878130793571472, 0.2561309039592743], [0.4053094983100891, -0.8762959241867065, 0.2373054176568985, -0.0074523985385894775, -0.15750834345817566]]

💡 forward_hook 可以完全替换某一层的输出，影响后续所有层


### 实验 5：full_backward_hook — 观察梯度流

In [11]:
# backward hook 是分析梯度流的重要工具
model = HookDemoModel()

grad_log = []

def log_grad(module, grad_input, grad_output):
    """记录梯度信息"""
    gi_info = []
    for i, g in enumerate(grad_input):
        if g is not None:
            gi_info.append(f'grad_in[{i}]: shape={list(g.shape)}, norm={g.norm().item():.4f}')
        else:
            gi_info.append(f'grad_in[{i}]: None')
    go_info = []
    for i, g in enumerate(grad_output):
        if g is not None:
            go_info.append(f'grad_out[{i}]: shape={list(g.shape)}, norm={g.norm().item():.4f}')
        else:
            go_info.append(f'grad_out[{i}]: None')
    
    grad_log.append({
        'module': type(module).__name__,
        'grad_input': gi_info,
        'grad_output': go_info,
    })

# 给所有 Linear 注册 backward hook
handles = []
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        handles.append(module.register_full_backward_hook(log_grad))

# 前向 + 反向
x = torch.randn(3, 10)
out = model(x)
loss = out.sum()
loss.backward()

print('=== 每层的梯度信息 ===')
for entry in grad_log:
    print(f'\n{entry["module"]}:')
    for gi in entry['grad_input']:
        print(f'  {gi}')
    for go in entry['grad_output']:
        print(f'  {go}')

# 清理
for h in handles:
    h.remove()

=== 每层的梯度信息 ===

Linear:
  grad_in[0]: shape=[3, 20], norm=2.4647
  grad_out[0]: shape=[3, 5], norm=3.8730

Linear:
  grad_in[0]: None
  grad_out[0]: shape=[3, 20], norm=1.3527


/tmp/claude-502/ipykernel_97234/766624894.py:37: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


### 实验 6：`handle.remove()` — Hook 的生命周期管理

In [12]:
# 演示 hook 的注册和移除
model = HookDemoModel()

# 注册 hook — handle 是 RemovableHandle 对象
call_count = 0
def count_calls(module, args, output):
    global call_count
    call_count += 1

handles = []
for module in model.modules():
    if isinstance(module, nn.Linear):
        h = module.register_forward_hook(count_calls)
        handles.append(h)

# 第一次 forward
x = torch.randn(3, 10)
out = model(x)
print(f'第一次 forward: {call_count} 次 hook 调用')
print(f'  注册的 handle 数量: {len(handles)}')
print(f'  fc1._forward_hooks 中的 hook 数: {len(model.fc1._forward_hooks)}')

# 移除部分 hook
handles[0].remove()  # 移除 fc1 的 hook
print(f'\n移除 handle[0] 后:')
print(f'  fc1._forward_hooks 中的 hook 数: {len(model.fc1._forward_hooks)}')

# 第二次 forward
call_count = 0
out = model(x)
print(f'第二次 forward: {call_count} 次 hook 调用')

# 移除剩余
for h in handles[1:]:
    h.remove()

第一次 forward: 2 次 hook 调用
  注册的 handle 数量: 2
  fc1._forward_hooks 中的 hook 数: 1

移除 handle[0] 后:
  fc1._forward_hooks 中的 hook 数: 0
第二次 forward: 1 次 hook 调用


### 实验 7：`always_call=True` — 即使 forward 抛异常也能执行

In [13]:
# always_call=True 的 hook 在 forward 抛出异常后仍然执行
class FailingModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 10)

    def forward(self, x):
        if x.sum() > 0:
            raise ValueError('模拟 forward 中的错误!')
        return self.linear(x)

model = FailingModule()

# 注册 always_call hook
always_log = []
def emergency_hook(module, args, output):
    always_log.append('always_call hook 被调用了!')
    print(f'  🚨 always_call hook 触发! output is None: {output is None}')

model.linear.register_forward_hook(emergency_hook, always_call=True)

x = torch.ones(3, 10)  # sum > 0 → 会触发异常
print('=== 执行会抛出异常的 forward ===')
try:
    out = model(x)
except ValueError as e:
    print(f'  ❌ 捕获异常: {e}')

print(f'\nalways_call hook 在异常后确实被执行了')
print(f'这可用于: 调试、资源清理、异常日志等场景')

=== 执行会抛出异常的 forward ===
  ❌ 捕获异常: 模拟 forward 中的错误!

always_call hook 在异常后确实被执行了
这可用于: 调试、资源清理、异常日志等场景


---

## Part D: 小结

### 要点清单

- [x] `model(x)` → `__call__` → `_wrapped_call_impl` → `_call_impl`
- [x] 无 hook 时走**快速路径**（直接 forward），有 hook 才走慢速路径
- [x] **forward_pre_hook**：在 forward 之前执行，可修改 args/kwargs
- [x] **forward_hook**：在 forward 之后执行，可修改 output，`always_call=True` 异常后也执行
- [x] **full_backward_hook**：反向传播时触发，可访问 grad_input 和 grad_output
- [x] **full_backward_pre_hook**：在计算模块梯度时提前触发，可修改 grad_output
- [x] `register_backward_hook` 已废弃，应使用 `register_full_backward_hook`
- [x] `handle.remove()` 随时可移除 hook，hook 存储在 `OrderedDict` 中

### Hook 执行顺序速查

```
正向: global_pre → local_pre → forward() → global_post → local_post
反向: ... ← backward_post ← backward_pre ← ...
```

### 与后续 Notebook 的关联

- **Notebook 08**（全局 Hook）：`register_module_forward_hook` 等全局版本，在局部 hook 之前执行
- **Notebook 07**（序列化）：serialization hooks 类似于 forward hooks 的注册模式
- **Notebook 10**（编译）：`torch.compile` 会替换 `_call_impl` 路径，hook 在编译模式下行为有差异

### 延伸阅读

- [PyTorch 源码 _call_impl](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py#L850)
- [PyTorch Hook 文档](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_hook)
- [Forward/Backward Hook 教程](https://pytorch.org/tutorials/beginner/former_torchies/nnft_tutorial.html)